# Activity 2: Window Functions in Pandas

**Module:** Week 6 Day 1
**Estimated Time:** 40 to 50 minutes
**Format:** Part 1 follows a simple dataset; Part 2 uses Snowflake datasets
**Prerequisites:** Activity 1 complete (SQL Window Functions)

## Objective

In SQL, window functions are powered by the `OVER` clause, combining `PARTITION BY`, `ORDER BY`, and frame definitions like `ROWS BETWEEN`. In Pandas, these exact same concepts exist but use different names and methods.

This activity is split into two parts:
1. **Part 1: The Basics** - We'll introduce the Pandas equivalents on a tiny dummy DataFrame, so you can clearly see how the mechanics work and match them to your SQL knowledge.
2. **Part 2: The Snowflake Challenge** - We'll load the datasets from Snowflake and you'll apply these Pandas methods to solve the exact same problems from the previous SQL activity.

Let's begin!

## Part 1: The Basics (Simple DataFrame)

First, let's create a tiny `demo_sales` dataset similar to the one we explored in the SQL window function demo.

In [ ]:
import pandas as pd

# Creating a tiny dummy DataFrame
data = {
    'STORE': ['downtown', 'downtown', 'downtown', 'downtown', 'airport', 'airport', 'airport', 'airport'],
    'DATE': ['2026-07-13', '2026-07-14', '2026-07-16', '2026-07-17', '2026-07-13', '2026-07-14', '2026-07-16', '2026-07-17'],
    'SALES': [200, 300, 250, 350, 400, 100, 500, 300]
}

demo_sales = pd.DataFrame(data)
# Always ensure dates are datetime types, and we MUST sort values!
demo_sales['DATE'] = pd.to_datetime(demo_sales['DATE'])
demo_sales = demo_sales.sort_values(by=['STORE', 'DATE']).reset_index(drop=True)

display(demo_sales)

### 1.1: `LAG()` / `LEAD()` equivalent is `.shift()`

In SQL: `LAG(sales) OVER (ORDER BY date)`
In Pandas, we use `.shift(1)`. 

**WARNING:** Pandas `.shift()` blindly shifts rows based on their *current order* in the DataFrame. If the DataFrame isn't sorted, your shift is garbage. That is why we sorted the DataFrame above.

In [ ]:
demo_sales['PREV_SALES'] = demo_sales['SALES'].shift(1)
demo_sales

### 1.2: Percentage Change equivalent is `.pct_change()`

Instead of manually doing `(current - previous) / previous`, Pandas has a built-in method `.pct_change()`.

In [ ]:
demo_sales['PCT_CHANGE'] = (demo_sales['SALES'].pct_change() * 100).round(2)
demo_sales

### 1.3: Window Frames (`ROWS BETWEEN`) equivalent is `.rolling()`

In SQL: `AVG(sales) OVER (ORDER BY date ROWS BETWEEN 1 PRECEDING AND CURRENT ROW)`
In Pandas, we use `.rolling()`.

A `.rolling(2)` frame looks at the current row and 1 preceding row.

In [ ]:
demo_sales['AVG_2DAY'] = demo_sales['SALES'].rolling(2).mean()
demo_sales

### Polars Spotlight: Window Expressions with `.over()`

In **Polars**, window functions are built directly into expressions using `.over()`, which mirrors SQL `OVER (PARTITION BY ...)` almost identically:

```python
import polars as pl

demo_pl = pl.DataFrame(demo_sales)

# Polars Window Function: LAG partitioned by store
demo_pl.with_columns(
    pl.col("SALES").shift(1).over("STORE").alias("PREV_SALES_POLARS"),
    pl.col("SALES").rolling_mean(window_size=2).over("STORE").alias("AVG_2DAY_POLARS")
)
```

Notice how `pl.col("SALES").shift(1).over("STORE")` directly matches SQL `LAG(sales) OVER (PARTITION BY store)`. This expression pattern is what you will encounter in **Snowpark** (`Window.partition_by`) and **PySpark** (`Window.partitionBy`).

### 1.4: Ranking & Running Totals equivalent is `.rank()`, `.cumsum()`, `.cummax()`

- `RANK() OVER (...)` -> `.rank(method="min")`
- `SUM(sales) OVER (...)` -> `.cumsum()`
- `MAX(sales) OVER (...)` -> `.cummax()`

In [ ]:
demo_sales['SALES_RANK'] = demo_sales['SALES'].rank(method="min", ascending=False)
demo_sales['RUNNING_TOTAL'] = demo_sales['SALES'].cumsum()
demo_sales

### 1.5: `PARTITION BY` equivalent is `.groupby()`

Look closely at the `PREV_SALES` column in the earlier step. Notice how the first row of 'downtown' shifted its value into the first row of 'airport'? 
This is because we shifted the entire column without partitioning! 

To replicate `PARTITION BY store`, we use `.groupby('STORE')` *before* calling the window function.

In [ ]:
# Notice how the first row of 'airport' now has a NaN for PREV_SALES, correctly bounding the partition!
demo_sales['PREV_SALES_PARTITIONED'] = demo_sales.groupby('STORE')['SALES'].shift(1)
demo_sales

## Part 2: The Snowflake Challenge

Now it's your turn. We will pull the exact same 3 tables from Snowflake that you queried in the SQL activity: `GOOGLE_STOCKS`, `CLOSING_PRICE`, and `MILK_PRODUCTION`.

Run the connection code below to load the dataframes.

In [ ]:
import pandas as pd
from configparser import ConfigParser
from snowflake import connector

# 1. Read config
config = ConfigParser()
config.read("snow.cfg")
params = dict(config["DEV"])

# 2. Connect
conn = connector.connect(**params)
cursor = conn.cursor()
print("connected as:", cursor.execute("SELECT CURRENT_USER()").fetchone()[0])

# 3. Load DataFrames
goog = cursor.execute("SELECT * FROM TECHCATALYST.STOCKS.GOOGLE_STOCKS ORDER BY DATE").fetch_pandas_all()
wide = cursor.execute("SELECT * FROM TECHCATALYST.STOCKS.CLOSING_PRICE ORDER BY DATE").fetch_pandas_all()
milk = cursor.execute("SELECT * FROM TECHCATALYST.STOCKS.MILK_PRODUCTION ORDER BY MONTH").fetch_pandas_all()

print(goog.shape, wide.shape, milk.shape)
goog.head()

### Challenge 1: Google Stocks

**Goal:** Create a 30-day moving average and a daily percentage change on the `goog` DataFrame.

1. Ensure the DataFrame is sorted by `DATE`.
2. Add a `PCT_CHANGE` column.
3. Add a `MA_30D` column using a 30-day rolling mean. (Hint: `min_periods=1` handles the edges like SQL).
4. Verify your best day (highest PCT_CHANGE) matches your SQL results!

In [ ]:
# YOUR CODE HERE
goog = goog.sort_values("DATE").reset_index(drop=True)




### Challenge 2: Closing Prices (Wide Table)

**Goal:** Reshape the `wide` table into a long format and calculate the daily change per symbol.

1. Use `.melt()` to reshape `wide` from wide to long (Columns: `DATE`, `SYMBOL`, `CLOSE`). Sort it by `SYMBOL` and `DATE`.
2. Calculate the daily difference (subtract previous day's close from current day's close) using `.groupby('SYMBOL')` and `.diff()` or `.shift(1)`. Assign it to `DAILY_CHANGE`.
3. Check that there are exactly 3 NaNs in the `DAILY_CHANGE` column (one for each symbol).

In [ ]:
# YOUR CODE HERE




### Challenge 3: Milk Production Seasonality

**Goal:** Calculate the Year-over-Year change.

1. Sort `milk` by `MONTH`.
2. Create a column `SAME_MONTH_LAST_YEAR` by lagging the `PRODUCTION` column by 12 periods.
3. Create a column `YOY_CHANGE` that is the difference between `PRODUCTION` and `SAME_MONTH_LAST_YEAR`.
4. Plot the production to see the seasonality!

In [ ]:
# YOUR CODE HERE




### Cleanup

In [ ]:
cursor.close()
conn.close()